###Ingest transactions parquet file   
1.read file using dataframe reader api   
2.add metadata columns - source file,ingestion timestamp     
3.write bronze tables using dataframe writer api

In [0]:
dbutils.widgets.text("p_batch_date","")
v_batch_date = dbutils.widgets.get("p_batch_date")

In [0]:
%run ../00-common/01.environment_config


In [0]:
%run ../00-common/02.bronze_functions

In [0]:
source_file = f"{raw_path}/{v_batch_date}/transactions/"

In [0]:
table_name = f"{catalog_name}.{bronze_schema}.transactions"

In [0]:
from pyspark.sql.types import *

In [0]:
txn_schema = StructType([
  StructField('TxnID', StringType()),
  StructField('BillerID', StringType()),
  StructField('CustomerID', StringType()),
  StructField('TxnAmount', DoubleType()),
  StructField('Status', StringType()),
  StructField('BillerRefID', StringType()),
  StructField('Timestamp', TimestampType())
])

In [0]:
txn_df = spark.read.format('parquet').load(source_file)

In [0]:
txn_audit = add_ingestion_metadata(txn_df)

In [0]:
txn_final= txn_audit.withColumn("batch_id", F.lit(v_batch_date))

In [0]:
txn_final.write.mode('overwrite').partitionBy('batch_id').option('replaceWhere',f"batch_id = '{v_batch_date}'").saveAsTable(table_name)

In [0]:
%sql
SELECT * FROM payment_app.bronze.transactions

TxnID,AccountNumber,BillerID,ConsumerNumber,TxnAmount,Status,RRN,BillerRefID,TransactionDate,ingestion_timestamp,source_file,batch_id
TXN1001,100001,KSEB,CONS001,1500.00,Success,RRN100001,BREF100001,2026-08-07T09:05:00.000Z,2026-08-24T09:42:11.500Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/transactions/data_7a084b7a-22ee-4428-8089-597b981a17ac_d095ba69-a0b7-46bc-8e25-c26a7624f195.parquet,2026-08-13
TXN1002,100002,WTR01,CONS002,800.00,Success,RRN100002,BREF100002,2026-08-07T09:15:00.000Z,2026-08-24T09:42:11.500Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/transactions/data_7a084b7a-22ee-4428-8089-597b981a17ac_d095ba69-a0b7-46bc-8e25-c26a7624f195.parquet,2026-08-13
TXN1003,100003,TEL01,CONS003,999.00,Pending,null,null,2026-08-07T09:30:00.000Z,2026-08-24T09:42:11.500Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/transactions/data_7a084b7a-22ee-4428-8089-597b981a17ac_d095ba69-a0b7-46bc-8e25-c26a7624f195.parquet,2026-08-13
TXN1004,100004,GAS01,CONS004,1200.00,Failed,null,null,2026-08-07T09:40:00.000Z,2026-08-24T09:42:11.500Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/transactions/data_7a084b7a-22ee-4428-8089-597b981a17ac_d095ba69-a0b7-46bc-8e25-c26a7624f195.parquet,2026-08-13
TXN1005,100005,DTH01,CONS005,450.00,Success,RRN100005,BREF100005,2026-08-07T09:50:00.000Z,2026-08-24T09:42:11.500Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/transactions/data_7a084b7a-22ee-4428-8089-597b981a17ac_d095ba69-a0b7-46bc-8e25-c26a7624f195.parquet,2026-08-13
TXN1006,100006,KSEB,CONS006,2300.00,Success,RRN100006,BREF100006,2026-08-07T10:05:00.000Z,2026-08-24T09:42:11.500Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/transactions/data_7a084b7a-22ee-4428-8089-597b981a17ac_d095ba69-a0b7-46bc-8e25-c26a7624f195.parquet,2026-08-13
TXN1007,100007,GAS01,CONS007,1750.00,Success,RRN100007,BREF100007,2026-08-07T10:20:00.000Z,2026-08-24T09:42:11.500Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/transactions/data_7a084b7a-22ee-4428-8089-597b981a17ac_d095ba69-a0b7-46bc-8e25-c26a7624f195.parquet,2026-08-13
TXN1008,100008,TEL01,CONS008,699.00,Success,RRN100008,BREF100008,2026-08-07T10:35:00.000Z,2026-08-24T09:42:11.500Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/transactions/data_7a084b7a-22ee-4428-8089-597b981a17ac_d095ba69-a0b7-46bc-8e25-c26a7624f195.parquet,2026-08-13
TXN1009,100009,WTR01,CONS009,650.00,Pending,null,null,2026-08-07T10:45:00.000Z,2026-08-24T09:42:11.500Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/transactions/data_7a084b7a-22ee-4428-8089-597b981a17ac_d095ba69-a0b7-46bc-8e25-c26a7624f195.parquet,2026-08-13
TXN1010,100010,DTH01,CONS010,399.00,Success,RRN100010,BREF100010,2026-08-07T11:00:00.000Z,2026-08-24T09:42:11.500Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/transactions/data_7a084b7a-22ee-4428-8089-597b981a17ac_d095ba69-a0b7-46bc-8e25-c26a7624f195.parquet,2026-08-13
